# 07 · Speech-to-Text (Multimodal Input)

The **Speech** service adds a voice input modality to the BrixoAI assistant. This satisfies the
capstone requirement for **multimodal AI pipelines** — the system now accepts:

| Modality | Input type | Service |
|---|---|---|
| Text | Typed message | Chat, all services |
| Image | Screenshot upload | UX Audit (vision) |
| Tabular | CSV / Excel upload | Data Insights |
| Audio | Voice recording | **This service** |

## Architecture

```
Audio file (mp3/wav/m4a/webm/etc.)
       │
       ▼
Groq Whisper large-v3
(state-of-the-art ASR model)
       │
       ▼
Transcript text
       │
       ├─── TRANSCRIBE ONLY ──► return text to user
       │
       └─── SPEECH + CHAT ───► inject into chat pipeline
                                    │
                                    ▼
                            LangChain agent response
```

## Why Groq Whisper large-v3?

- **Best-in-class accuracy** across 99+ languages
- **Groq inference** provides ~10× faster transcription vs OpenAI API
- Supports all common audio formats: mp3, wav, m4a, webm, ogg, flac
- Max file size: 25 MB per request (sufficient for meeting recordings up to ~30 min)

In [1]:
%pip install -q groq langchain-groq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, io, asyncio
from pathlib import Path
from dotenv import load_dotenv
import groq
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv(dotenv_path=Path('..') / '.env')

WHISPER_MODEL = 'whisper-large-v3'
SUPPORTED_FORMATS = ['mp3', 'mp4', 'mpeg', 'mpga', 'm4a', 'wav', 'webm', 'ogg', 'flac']

groq_client = groq.AsyncGroq(api_key=os.environ['GROQ_API_KEY'])
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.4)

print('Whisper model:', WHISPER_MODEL)
print('Supported formats:', SUPPORTED_FORMATS)

Whisper model: whisper-large-v3
Supported formats: ['mp3', 'mp4', 'mpeg', 'mpga', 'm4a', 'wav', 'webm', 'ogg', 'flac']


## Transcription Function

The core function sends an audio file to Groq's Whisper API and returns the transcript.

In [3]:
async def transcribe_audio(
    audio_bytes: bytes,
    filename: str = 'audio.mp3',
    language: str | None = None,
) -> dict:
    """
    Transcribe audio bytes using Groq Whisper.

    Args:
        audio_bytes: raw audio file content
        filename: filename with extension (determines format)
        language: ISO 639-1 language code (e.g. 'en', 'fr', 'ar')
                  or None for automatic detection

    Returns:
        dict with 'text' (transcript) and 'language' (detected/specified)
    """
    ext = Path(filename).suffix.lstrip('.').lower()
    if ext not in SUPPORTED_FORMATS:
        raise ValueError(f'Unsupported format: .{ext}. Supported: {SUPPORTED_FORMATS}')

    # Map extension to MIME type for the API call
    mime_map = {
        'mp3': 'audio/mpeg', 'mp4': 'audio/mp4', 'mpeg': 'audio/mpeg',
        'mpga': 'audio/mpeg', 'm4a': 'audio/mp4', 'wav': 'audio/wav',
        'webm': 'audio/webm', 'ogg': 'audio/ogg', 'flac': 'audio/flac',
    }
    content_type = mime_map.get(ext, 'audio/mpeg')

    kwargs = {
        'file': (filename, audio_bytes, content_type),
        'model': WHISPER_MODEL,
        'response_format': 'verbose_json',
    }
    if language:
        kwargs['language'] = language

    response = await groq_client.audio.transcriptions.create(**kwargs)

    return {
        'text': response.text.strip(),
        'language': getattr(response, 'language', language or 'auto'),
        'duration_s': getattr(response, 'duration', None),
    }

print('transcribe_audio() defined')
print('Usage: result = await transcribe_audio(audio_bytes, filename="clip.mp3")')

transcribe_audio() defined
Usage: result = await transcribe_audio(audio_bytes, filename="clip.mp3")


## Generate a Test Audio Clip

We use macOS `say` or Windows TTS to create a small test clip, or alternatively
write raw silence (valid WAV) to test the pipeline without requiring a real recording.

In [4]:
import wave, struct, tempfile, subprocess, platform

def generate_silent_wav(duration_s: float = 1.0, sample_rate: int = 16000) -> bytes:
    """Generate a minimal valid silent WAV file in memory."""
    num_samples = int(duration_s * sample_rate)
    buffer = io.BytesIO()
    with wave.open(buffer, 'w') as wf:
        wf.setnchannels(1)       # mono
        wf.setsampwidth(2)       # 16-bit
        wf.setframerate(sample_rate)
        wf.writeframes(struct.pack('<' + 'h' * num_samples, *([0] * num_samples)))
    return buffer.getvalue()

# NOTE: If you have a real audio file, load it instead:
# with open('my_recording.mp3', 'rb') as f:
#     audio_bytes = f.read()

audio_bytes = generate_silent_wav(2.0)
print(f'Test WAV size: {len(audio_bytes)} bytes ({len(audio_bytes)/1024:.1f} KB)')

Test WAV size: 64044 bytes (62.5 KB)


In [5]:
# Transcribe the audio (silent audio will produce an empty or [BLANK_AUDIO] transcript)
try:
    result = await transcribe_audio(audio_bytes, filename='test.wav', language='en')
    print('Transcript:  ', result['text'] or '(empty — silent audio)')
    print('Language:    ', result['language'])
    print('Duration:    ', result['duration_s'], 's')
except Exception as e:
    print(f'API error: {e}')
    print('Tip: Try a real audio file for a meaningful transcript.')

Transcript:   you
Language:     English
Duration:     2 s


## Transcribe + Chat Pipeline

After transcription, the text is passed through the full LangChain agent so the user
gets an **AI response** to their spoken message, not just the raw transcript.

In [6]:
CHAT_SYSTEM_PROMPT = """
You are BrixoAI Assistant. Answer helpfully and concisely.
The user's message was transcribed from speech — treat it as a natural conversational query.
""".strip()

async def speech_to_chat(audio_bytes: bytes, filename: str = 'audio.wav', language: str | None = None) -> dict:
    """
    Transcribe audio then generate an AI response.

    Returns:
        {
            'transcript': str,  # what the user said
            'response': str,    # AI reply
            'language': str,
        }
    """
    # Step 1: Transcribe
    transcription = await transcribe_audio(audio_bytes, filename, language)
    transcript = transcription['text']

    if not transcript.strip():
        return {
            'transcript': '',
            'response': "I couldn't hear anything in the recording. Please try again.",
            'language': transcription['language'],
        }

    # Step 2: Chat
    display_text = f'🎙️ *"{transcript}"*\n\n'

    ai_msg = await llm.ainvoke([
        SystemMessage(content=CHAT_SYSTEM_PROMPT),
        HumanMessage(content=transcript),
    ])

    return {
        'transcript': transcript,
        'formatted_display': display_text + ai_msg.content,
        'response': ai_msg.content,
        'language': transcription['language'],
    }

print('speech_to_chat() defined')

speech_to_chat() defined


## Simulate Speech Input with a Known Transcript

Since generating real audio in a notebook is OS-dependent, we test the *chat* leg
of the pipeline directly with a mock transcript.

In [7]:
# Directly test the chat leg with a simulated transcript
simulated_transcripts = [
    'What services does BrixoAI offer?',
    'I want to audit my website can you help?',
    'Tell me about your data analytics service',
]

for t in simulated_transcripts:
    ai_msg = await llm.ainvoke([
        SystemMessage(content=CHAT_SYSTEM_PROMPT),
        HumanMessage(content=t),
    ])
    print(f'Speech: "{t}"')
    print(f'Response: {ai_msg.content[:300]}')
    print('---')

Speech: "What services does BrixoAI offer?"
Response: BrixoAI offers a range of services, including text-based conversational AI, language translation, text summarization, and content generation. We can assist with tasks such as answering questions, providing information, and helping with writing and research. Our goal is to provide helpful and accurat
---
Speech: "I want to audit my website can you help?"
Response: I'd be happy to help you audit your website. What specific areas of your website would you like to audit (e.g. performance, SEO, security, user experience)? Or would you like me to suggest a general checklist to get started?
---
Speech: "Tell me about your data analytics service"
Response: Our data analytics service helps organizations make data-driven decisions by collecting, analyzing, and interpreting complex data. We offer tools and expertise to uncover insights, identify trends, and optimize business performance. What specific aspects of data analytics are you interest

## Language Detection Demo

Whisper large-v3 auto-detects language when no language code is specified.
Here we show how to pass explicit language codes when the source language is known.

In [8]:
LANGUAGE_CODES = {
    'Auto-detect':  None,
    'English':      'en',
    'French':       'fr',
    'Spanish':      'es',
    'Arabic':       'ar',
    'Portuguese':   'pt',
    'German':       'de',
    'Chinese':      'zh',
    'Japanese':     'ja',
}

print('Supported language hint codes:')
for name, code in LANGUAGE_CODES.items():
    print(f'  {name:15s} → {code or "(auto)"}')

Supported language hint codes:
  Auto-detect     → (auto)
  English         → en
  French          → fr
  Spanish         → es
  Arabic          → ar
  Portuguese      → pt
  German          → de
  Chinese         → zh
  Japanese        → ja


## File Size Validation

The API enforces a 25 MB limit. We validate on the server side before sending.

In [9]:
MAX_AUDIO_BYTES = 25 * 1024 * 1024  # 25 MB

def validate_audio_file(audio_bytes: bytes, filename: str) -> None:
    """Raise ValueError if the file is too large or the format is unsupported."""
    ext = Path(filename).suffix.lstrip('.').lower()
    if ext not in SUPPORTED_FORMATS:
        raise ValueError(f'Unsupported format: .{ext}')
    if len(audio_bytes) > MAX_AUDIO_BYTES:
        mb = len(audio_bytes) / 1024 / 1024
        raise ValueError(f'File too large: {mb:.1f} MB (max 25 MB)')
    if len(audio_bytes) == 0:
        raise ValueError('Empty audio file')

# Test the validator
test_cases = [
    (audio_bytes, 'test.wav', 'Should pass'),
    (b'data', 'clip.txt', 'Should fail — unsupported format'),
    (b'', 'clip.mp3', 'Should fail — empty file'),
]

for data, name, label in test_cases:
    try:
        validate_audio_file(data, name)
        print(f'[PASS] {name} — {label}')
    except ValueError as e:
        print(f'[FAIL] {name} — {e}')

[PASS] test.wav — Should pass
[FAIL] clip.txt — Unsupported format: .txt
[FAIL] clip.mp3 — Empty audio file


## Integration with FastAPI

In the full application, the `/speech/transcribe` and `/speech/chat` endpoints
accept `UploadFile` objects and pipe bytes through the same functions above.

```python
@router.post('/speech/transcribe')
async def transcribe_endpoint(file: UploadFile):
    audio_bytes = await file.read()
    validate_audio_file(audio_bytes, file.filename)
    result = await transcribe_audio(audio_bytes, file.filename)
    return result

@router.post('/speech/chat')
async def speech_chat_endpoint(file: UploadFile, language: str | None = None):
    audio_bytes = await file.read()
    validate_audio_file(audio_bytes, file.filename)
    return await speech_to_chat(audio_bytes, file.filename, language)
```

The Streamlit UI renders the transcript with a 🎙️ prefix before the AI response.

## Summary

- **Groq Whisper large-v3** provides the highest public benchmark accuracy (WER) for speech transcription.
- The two-stage pipeline (transcribe → chat) means voice input is a **transparent wrapper** — all downstream
  services work unchanged with transcribed text.
- **Language hinting** improves accuracy for known non-English inputs.
- **File validation** (format + size + non-empty) prevents unnecessary API calls and returns clear error messages.
- **Multimodal capability**: BrixoAI Assistant now accepts text, images, tabular data, and audio — covering four distinct input modalities.